# NB-14: FramePack Multi-Scale Patchify and Per-Token RoPE

## Learning Objectives
- Understand how FramePackMotioner splits motion frames into 3 buckets (1x/2x/4x) and projects each through a Conv3d at different kernel/stride sizes
- Trace the reversed split order where oldest frames get the coarsest (4x) representation and newest frames get the finest (1x)
- See how S2V's `rope_precompute` assigns per-token positions via `np.linspace` with negative temporal offsets, connecting back to NB-03's foreshadowing

## Prerequisites
- **Prior notebooks:** NB-03 (3D RoPE, especially the S2V comparison in cells 13-14), NB-06 (patchify concept)
- **Assumed concepts:** Conv3d kernel/stride mechanics, complex number RoPE representation

## Concept Map
- Motion reference frames -> **FramePackMotioner** (3-scale patchify) -> per-token RoPE -> motion tokens injected into unified sequence in NB-16
- NB-03 foreshadowed S2V's `rope_precompute` using `np.linspace` vs integer indices -- this notebook traces the full mechanism

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios (up to 6 levels up).
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for S2V demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py (base DiT, needed for precompute_freqs_cis_3d)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Load wan_video_dit_s2v.py (S2V model with FramePackMotioner and rope_precompute)
_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"
_spec_s2v = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)
s2v = importlib.util.module_from_spec(_spec_s2v)
sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v
_spec_s2v.loader.exec_module(s2v)

# Import symbols used in this notebook
from diffsynth.models.wan_video_dit_s2v import FramePackMotioner, rope_precompute
from diffsynth.models.wan_video_dit import precompute_freqs_cis_3d
import numpy as np
import torch
import torch.nn as nn

print("Setup complete.")

## FramePack Multi-Scale Motion Encoding

```
FramePack Multi-Scale Motion Encoding
======================================

INPUT: 7 motion frames (zip_frame_buckets=[1, 2, 4] -> sum=7)
       each frame: [16, H, W] = [16, 16, 16] latent

SPLIT (REVERSED -- oldest frames first, newest last):
  Frames:  [f0, f1, f2, f3] | [f4, f5] | [f6]
           \_____________/    \_______/   \__/
           4x bucket (oldest)  2x bucket   1x bucket (newest)
           coarsest            medium       finest

           ^--- recent motion gets finer spatial representation

MULTI-SCALE PATCHIFY (separate Conv3d per bucket):
  4x: Conv3d(k=4,8,8, s=4,8,8)  [1, 16, 4, 16, 16] -> [1, 384, 1, 2, 2]  -> flatten -> [1,  4, 384]
  2x: Conv3d(k=2,4,4, s=2,4,4)  [1, 16, 2, 16, 16] -> [1, 384, 1, 4, 4]  -> flatten -> [1, 16, 384]
  1x: Conv3d(k=1,2,2, s=1,2,2)  [1, 16, 1, 16, 16] -> [1, 384, 1, 8, 8]  -> flatten -> [1, 64, 384]

CONCATENATE (1x first, then 2x, then 4x):
  motion_lat = cat(1x, 2x, 4x) = [1, 84, 384]  (64 + 16 + 4 = 84 tokens)

PER-TOKEN ROPE (negative temporal offsets):
  1x bucket: start_time_id = -1   (most recent)
  2x bucket: start_time_id = -3   (medium past)
  4x bucket: start_time_id = -7   (distant past)

  Negative offset -> conj() flips rotation direction -> past-pointing positions
```

In [ ]:
# Reduced config (consistent with all prior phases)
dim = 384
num_heads = 4
head_dim = dim // num_heads  # 96
patch_size = (1, 2, 2)
in_dim = 16  # VAE latent channels

# S2V-specific config
zip_frame_buckets = [1, 2, 4]  # reduced from production [1, 2, 16]
total_frames = sum(zip_frame_buckets)  # 7

# Spatial dimensions (16x16, divisible by 8 for 4x bucket)
lat_height, lat_width = 16, 16

print(f"zip_frame_buckets: {zip_frame_buckets}  (total: {total_frames} frames)")
print(f"Spatial: {lat_height}x{lat_width}")
print(f"dim={dim}, num_heads={num_heads}, head_dim={head_dim}")

## 1. FramePackMotioner -- Three-Scale Patchify

FramePackMotioner uses three separate Conv3d projections, each at a different kernel/stride size,
to convert motion reference frames into multi-scale tokens:

| Sub-module | Type | Kernel/Stride | Input Shape | Output Shape | Tokens |
|------------|------|---------------|-------------|-------------|--------|
| `proj` (1x) | Conv3d | (1,2,2)/(1,2,2) | [1,16,1,16,16] | [1,384,1,8,8] | 64 |
| `proj_2x` (2x) | Conv3d | (2,4,4)/(2,4,4) | [1,16,2,16,16] | [1,384,1,4,4] | 16 |
| `proj_4x` (4x) | Conv3d | (4,8,8)/(4,8,8) | [1,16,4,16,16] | [1,384,1,2,2] | 4 |

Compare with NB-06's single-scale patchify (`Conv3d(16, 384, (1,2,2), (1,2,2))`):
FramePackMotioner extends the same concept to three scales, trading temporal compression
for spatial detail in each bucket.

In [ ]:
# Source: wan_video_dit_s2v.py, line 171
packer = FramePackMotioner(
    inner_dim=dim,
    num_heads=num_heads,
    zip_frame_buckets=zip_frame_buckets,
    drop_mode="padd",
)
total_params = sum(p.numel() for p in packer.parameters())
print(f"FramePackMotioner: {total_params:,} parameters")
print(f"  zip_frame_buckets: {packer.zip_frame_buckets.tolist()}")

# Verify Conv3d kernels and strides
assert packer.proj.kernel_size == (1, 2, 2)
assert packer.proj.stride == (1, 2, 2)
print(f"\nproj (1x):    kernel={packer.proj.kernel_size}, stride={packer.proj.stride}")

assert packer.proj_2x.kernel_size == (2, 4, 4)
assert packer.proj_2x.stride == (2, 4, 4)
print(f"proj_2x (2x): kernel={packer.proj_2x.kernel_size}, stride={packer.proj_2x.stride}")

assert packer.proj_4x.kernel_size == (4, 8, 8)
assert packer.proj_4x.stride == (4, 8, 8)
print(f"proj_4x (4x): kernel={packer.proj_4x.kernel_size}, stride={packer.proj_4x.stride}")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 186-200
# Simulate motion latents (e.g., 5 available frames, padded to 7)
motion_frames_available = 5
motion_input = [torch.randn(in_dim, motion_frames_available, lat_height, lat_width)]
print(f"Motion input: {motion_input[0].shape}  ({motion_frames_available} available frames)")

# Manual padding to zip_frame_buckets.sum() = 7 frames
padd_lat = torch.zeros(in_dim, total_frames, lat_height, lat_width)
overlap_frame = min(padd_lat.shape[1], motion_input[0].shape[1])
padd_lat[:, -overlap_frame:] = motion_input[0][:, -overlap_frame:]
padd_lat = padd_lat.unsqueeze(0)  # add batch dim
print(f"After padding: {padd_lat.shape}  (padded to {total_frames} frames)")
assert padd_lat.shape == torch.Size([1, in_dim, total_frames, lat_height, lat_width])

print(f"\nPadding fills from the END: frames 0-{total_frames - overlap_frame - 1} are zeros, "
      f"frames {total_frames - overlap_frame}-{total_frames - 1} are real data")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 201-203
# CRITICAL: split uses [::-1] -- REVERSED bucket order
reversed_buckets = list(packer.zip_frame_buckets)[::-1]
print(f"zip_frame_buckets: {packer.zip_frame_buckets.tolist()}")
print(f"Reversed for split: {reversed_buckets}  (4x first, then 2x, then 1x)")

clean_latents_4x, clean_latents_2x, clean_latents_post = padd_lat[
    :, :, -total_frames:, :, :
].split(reversed_buckets, dim=2)

print(f"\n4x bucket (oldest, coarsest):  {clean_latents_4x.shape}  (frames 0-3)")
print(f"2x bucket (medium):            {clean_latents_2x.shape}  (frames 4-5)")
print(f"1x bucket (newest, finest):    {clean_latents_post.shape}  (frame 6)")
assert clean_latents_4x.shape == torch.Size([1, in_dim, 4, lat_height, lat_width])
assert clean_latents_2x.shape == torch.Size([1, in_dim, 2, lat_height, lat_width])
assert clean_latents_post.shape == torch.Size([1, in_dim, 1, lat_height, lat_width])

print(f"\nKey insight: recent motion gets finer spatial representation")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 206-208
packer.eval()
with torch.no_grad():
    # 1x scale: Conv3d(k=1,2,2) -> finest spatial resolution
    tokens_1x = packer.proj(clean_latents_post).flatten(2).transpose(1, 2)
    print(f"1x: {clean_latents_post.shape} -> proj -> flatten -> {tokens_1x.shape}")
    assert tokens_1x.shape == torch.Size([1, 64, dim])  # 1*8*8 = 64 tokens

    # 2x scale: Conv3d(k=2,4,4) -> medium spatial resolution
    tokens_2x = packer.proj_2x(clean_latents_2x).flatten(2).transpose(1, 2)
    print(f"2x: {clean_latents_2x.shape} -> proj_2x -> flatten -> {tokens_2x.shape}")
    assert tokens_2x.shape == torch.Size([1, 16, dim])  # 1*4*4 = 16 tokens

    # 4x scale: Conv3d(k=4,8,8) -> coarsest spatial resolution
    tokens_4x = packer.proj_4x(clean_latents_4x).flatten(2).transpose(1, 2)
    print(f"4x: {clean_latents_4x.shape} -> proj_4x -> flatten -> {tokens_4x.shape}")
    assert tokens_4x.shape == torch.Size([1, 4, dim])  # 1*2*2 = 4 tokens

    # Concatenate: 1x first (finest), then 2x, then 4x (coarsest)
    motion_lat = torch.cat([tokens_1x, tokens_2x, tokens_4x], dim=1)
    print(f"\nConcatenated: {motion_lat.shape}  (64 + 16 + 4 = {64 + 16 + 4} tokens)")
    assert motion_lat.shape == torch.Size([1, 84, dim])

print(f"\nToken distribution: 1x={tokens_1x.shape[1]}, 2x={tokens_2x.shape[1]}, 4x={tokens_4x.shape[1]}")

### Token Count Breakdown

| Bucket | Frames | Conv3d | Output Grid | Tokens |
|--------|--------|--------|-------------|--------|
| 1x (newest) | 1 | (1,2,2)/(1,2,2) | 1 x 8 x 8 | 64 |
| 2x | 2 | (2,4,4)/(2,4,4) | 1 x 4 x 4 | 16 |
| 4x (oldest) | 4 | (4,8,8)/(4,8,8) | 1 x 2 x 2 | 4 |
| **Total** | **7** | | | **84** |

The newest frame gets 64 tokens (fine detail), while the 4 oldest frames share only 4 tokens (coarse summary).
This is the FramePack insight: allocate more spatial resolution to recent motion.

## 2. Per-Token RoPE -- Connecting Back to NB-03

In NB-03, we saw the structural difference between base DiT grid RoPE and S2V per-token RoPE:
- **Base DiT:** positions are sequential integers (0, 1, 2, ...) computed via `precompute_freqs_cis_3d`
- **S2V:** positions are sampled via `np.linspace` with negative temporal offsets

Now we trace the full mechanism. Each FramePack bucket gets its own `grid_sizes` with
a negative `start_time_id`, because motion frames represent PAST context relative to
the content frames at position 0.

In [ ]:
# Source: wan_video_dit_s2v.py, lines 217-243
# Compute grid_sizes exactly as FramePackMotioner.forward does

# 1x bucket: start_time_id = -sum(buckets[:1]) = -1
start_1x = -int(packer.zip_frame_buckets[:1].sum())
end_1x = start_1x + int(packer.zip_frame_buckets[0])
grid_sizes_1x = [
    [torch.tensor([start_1x, 0, 0]).unsqueeze(0),
     torch.tensor([end_1x, lat_height // 2, lat_width // 2]).unsqueeze(0),
     torch.tensor([int(packer.zip_frame_buckets[0]), lat_height // 2, lat_width // 2]).unsqueeze(0)]
]
print(f"1x bucket: start_time_id={start_1x}, end_time_id={end_1x}")
print(f"  spatial grid: {lat_height//2}x{lat_width//2} = 8x8")
print(f"  total tokens: 1 * 8 * 8 = 64")

# 2x bucket: start_time_id = -sum(buckets[:2]) = -3
start_2x = -int(packer.zip_frame_buckets[:2].sum())
end_2x = start_2x + int(packer.zip_frame_buckets[1]) // 2
grid_sizes_2x = [
    [torch.tensor([start_2x, 0, 0]).unsqueeze(0),
     torch.tensor([end_2x, lat_height // 4, lat_width // 4]).unsqueeze(0),
     torch.tensor([int(packer.zip_frame_buckets[1]), lat_height // 2, lat_width // 2]).unsqueeze(0)]
]
print(f"\n2x bucket: start_time_id={start_2x}, end_time_id={end_2x}")
print(f"  spatial grid: {lat_height//4}x{lat_width//4} = 4x4")
print(f"  total tokens: 1 * 4 * 4 = 16")

# 4x bucket: start_time_id = -sum(buckets[:3]) = -7
start_4x = -int(packer.zip_frame_buckets[:3].sum())
end_4x = start_4x + int(packer.zip_frame_buckets[2]) // 4
grid_sizes_4x = [
    [torch.tensor([start_4x, 0, 0]).unsqueeze(0),
     torch.tensor([end_4x, lat_height // 8, lat_width // 8]).unsqueeze(0),
     torch.tensor([int(packer.zip_frame_buckets[2]), lat_height // 2, lat_width // 2]).unsqueeze(0)]
]
print(f"\n4x bucket: start_time_id={start_4x}, end_time_id={end_4x}")
print(f"  spatial grid: {lat_height//8}x{lat_width//8} = 2x2")
print(f"  total tokens: 1 * 2 * 2 = 4")

grid_sizes_all = grid_sizes_1x + grid_sizes_2x + grid_sizes_4x
print(f"\nAll grid_sizes: {len(grid_sizes_all)} buckets")
assert len(grid_sizes_all) == 3

### RoPE Grid Sizes (Negative Temporal Offsets)

| Bucket | start_time_id | end_time_id | Spatial Grid | Tokens | Temporal Position |
|--------|---------------|-------------|-------------|--------|-------------------|
| 1x (newest) | -1 | 0 | 8 x 8 | 64 | Most recent |
| 2x | -3 | -2 | 4 x 4 | 16 | Medium past |
| 4x (oldest) | -7 | -6 | 2 x 2 | 4 | Distant past |

**Why negative?** Content tokens occupy temporal position 0+. Motion tokens represent
PAST context, so they get negative offsets. The `conj()` operation in `rope_precompute`
reverses the rotation direction for these negative positions.

In [ ]:
# Source: wan_video_dit_s2v.py, lines 26-82
# Trace how rope_precompute handles negative start_time_id

# Prepare input tensor for RoPE
x_rope = motion_lat.detach().view(1, 84, num_heads, head_dim)
print(f"Input to rope_precompute: {x_rope.shape}")

# Use FramePackMotioner's precomputed frequencies
freqs = packer.freqs
print(f"Frequencies: {freqs.shape}")

# Call rope_precompute
motion_rope = rope_precompute(x_rope, grid_sizes_all, freqs, start=None)
print(f"Motion RoPE output: {motion_rope.shape}")
assert motion_rope.shape == torch.Size([1, 84, num_heads, head_dim // 2])

# Source: wan_video_dit_s2v.py, lines 58-61
# For negative f_o: f_sam = np.linspace(-f_o, (-t_f - f_o) + 1, seq_f)
# Then: freqs[0][f_sam].conj() -- conjugation reverses rotation
print(f"\nnp.linspace sampling (temporal dimension):")

# 1x: f_o=-1, t_f=1, seq_f=end_1x-start_1x=1
# linspace(-(-1), (-1-(-1))+1, 1) = linspace(1, 1, 1) = [1]
seq_f_1x = end_1x - start_1x
t_f_1x = int(packer.zip_frame_buckets[0])
f_sam_1x = np.linspace(-start_1x, (-t_f_1x - start_1x) + 1, seq_f_1x).astype(int).tolist()
print(f"  1x: np.linspace({-start_1x}, {(-t_f_1x - start_1x) + 1}, {seq_f_1x}) = {f_sam_1x}  (then conj())")

# 2x: f_o=-3, t_f=2, seq_f=end_2x-start_2x=1
# linspace(-(-3), (-2-(-3))+1, 1) = linspace(3, 2, 1) = [3]
seq_f_2x = end_2x - start_2x
t_f_2x = int(packer.zip_frame_buckets[1])
f_sam_2x = np.linspace(-start_2x, (-t_f_2x - start_2x) + 1, seq_f_2x).astype(int).tolist()
print(f"  2x: np.linspace({-start_2x}, {(-t_f_2x - start_2x) + 1}, {seq_f_2x}) = {f_sam_2x}  (then conj())")

# 4x: f_o=-7, t_f=4, seq_f=end_4x-start_4x=1
# linspace(-(-7), (-4-(-7))+1, 1) = linspace(7, 4, 1) = [7]
seq_f_4x = end_4x - start_4x
t_f_4x = int(packer.zip_frame_buckets[2])
f_sam_4x = np.linspace(-start_4x, (-t_f_4x - start_4x) + 1, seq_f_4x).astype(int).tolist()
print(f"  4x: np.linspace({-start_4x}, {(-t_f_4x - start_4x) + 1}, {seq_f_4x}) = {f_sam_4x}  (then conj())")

print(f"\nKey: negative start_time_id triggers freqs[f_sam].conj()")
print(f"Conjugation reverses rotation direction -> past-pointing temporal positions")

In [ ]:
# Source: wan_video_dit.py, line 75 (base) vs wan_video_dit_s2v.py, line 26 (S2V)
# Contrast the two RoPE approaches

# Base DiT: integer grid positions
print("=== Base DiT Grid RoPE ===")
print(f"  Positions: 0, 1, 2, ... (sequential integers)")
print(f"  All tokens get uniform temporal spacing")
print(f"  Grid: F x H x W with start at (0, 0, 0)")

print(f"\n=== S2V Per-Token RoPE ===")
print(f"  Positions: sampled via np.linspace(start, end, num_tokens)")
print(f"  Different buckets get different temporal ranges")
print(f"  Negative start_time_id for motion (past context)")
print(f"  Positive positions 0+ for content (current denoising target)")

# Show the actual position assignments
print(f"\nS2V temporal positions:")
print(f"  Content tokens: 0, 1, 2, 3 (positive, present)")
print(f"  1x motion:      start_time_id = {start_1x} (nearest past)")
print(f"  2x motion:      start_time_id = {start_2x} (medium past)")
print(f"  4x motion:      start_time_id = {start_4x} (distant past)")
print(f"\nThis connects to NB-03's foreshadowing about S2V using np.linspace vs integer indices")

In [ ]:
# Source: wan_video_dit_s2v.py, line 185
# Run the complete FramePackMotioner.forward and verify outputs
with torch.no_grad():
    mot_list, mot_rope_list = packer(motion_input, add_last_motion=2)

assert len(mot_list) == 1, f"Expected 1 motion tensor, got {len(mot_list)}"
assert len(mot_rope_list) == 1, f"Expected 1 rope tensor, got {len(mot_rope_list)}"

mot_out = mot_list[0]
mot_rope_out = mot_rope_list[0]

print(f"FramePackMotioner output:")
print(f"  motion_lat: {mot_out.shape}")
print(f"  motion_rope: {mot_rope_out.shape}")
assert mot_out.shape == torch.Size([1, 84, dim]), f"Expected [1, 84, {dim}], got {mot_out.shape}"
assert mot_rope_out.shape == torch.Size([1, 84, num_heads, head_dim // 2])

# Verify token counts match our manual computation
assert mot_out.shape[1] == 64 + 16 + 4, f"Expected 84 tokens (64+16+4), got {mot_out.shape[1]}"
print(f"\nVerified: 84 tokens = 64 (1x) + 16 (2x) + 4 (4x)")
print(f"FramePackMotioner forward matches manual trace")

## Summary

### Key Takeaways
- **Reversed split order:** FramePackMotioner splits motion frames in REVERSE order -- oldest frames go to 4x (coarsest), newest to 1x (finest), allocating more spatial detail to recent motion
- **Multi-scale Conv3d:** Three Conv3d projections at different kernel/stride sizes produce a multi-scale token representation: 64 + 16 + 4 = 84 tokens for our reduced config
- **Per-token RoPE:** S2V uses `np.linspace` with negative temporal offsets so motion tokens encode past context, with `conj()` reversing rotation direction for negative positions

### Source References

| Symbol | Location |
|--------|----------|
| FramePackMotioner | `wan_video_dit_s2v.py`, line 171 |
| FramePackMotioner.forward | `wan_video_dit_s2v.py`, line 185 |
| rope_precompute | `wan_video_dit_s2v.py`, line 26 |
| precompute_freqs_cis_3d | `wan_video_dit.py`, line 75 |

### Next
- NB-15 shows the WanS2VDiTBlock (dual-timestep) and AudioInjector
- NB-16 shows how motion tokens are injected into the unified sequence

## Exercises

### Exercise 1 -- Production token count
With the production config `zip_frame_buckets=[1, 2, 16]`, how many total tokens would FramePackMotioner produce at 16x16 spatial? At what spatial resolution would the 4x bucket tokens be? (Hint: 16 frames through `Conv3d(k=4,8,8)`.)

### Exercise 2 -- Why reversed split?
Why does the split use `[::-1]` (reverse) order? What would happen if frames were split in the forward order `[1, 2, 4]` instead? Which frames would get the finest representation?

### Exercise 3 -- Temporal stride division
The 2x bucket's `end_time_id` is computed as `start_time_id + bucket_size // 2`. Why divide by 2? (Hint: the Conv3d stride in the temporal dimension is 2 for `proj_2x`, so 2 input frames become 1 temporal token.)

## Coming Up

We now have the two input streams that feed into the S2V transformer:
- **Audio embeddings** (NB-13): global + local features from CausalAudioEncoder
- **Motion tokens** (this notebook): 84 multi-scale tokens from FramePackMotioner

In NB-15, we'll see how the transformer block processes these: WanS2VDiTBlock applies
dual-timestep modulation (real timestep for content, zero for ref/motion), and
AudioInjector performs per-frame cross-attention after specified blocks.